# SignBridge - Model Training & Inference

In [1]:
## Step 1: Load and Inspect One Sample

from pathlib import Path
import numpy as np
import pandas as pd

# Paths
TRAIN_CSV = Path('..') / 'data' / '(final)_how2sign_train_filtered.csv'
KEYPOINTS_DIR = Path('..') / 'data' / 'keypoints_preprocessed' / 'train'

# Load CSV
df = pd.read_csv(TRAIN_CSV, sep='\t')
print(f'Total training examples: {len(df)}')
print(f'Columns: {list(df.columns)}')

# Pick first example
sample_row = df.sample(1).iloc[0]
sentence_name = sample_row['SENTENCE_NAME']
sentence_text = sample_row['SENTENCE']
duration_sec = sample_row['duration_sec']

print(f'\nSample: {sentence_name}')
print(f'Text: "{sentence_text}"')
print(f'Duration: {duration_sec:.2f}s')

# Load keypoints
keypoint_path = KEYPOINTS_DIR / f'{sentence_name}.npz'

if keypoint_path.exists():
    data = np.load(keypoint_path)
    keypoints = data['keypoints']  # (T, N, 3)
    mask = data['mask']  # (T, N)
    
    T, N, C = keypoints.shape
    
    print(f'\nKeypoint file: {keypoint_path.name}')
    print(f'  Shape: (T={T} frames, N={N} landmarks, C={C} coords)')
    print(f'  Valid landmarks: {mask.sum()} / {mask.size} ({mask.sum()/mask.size*100:.1f}%)')
    print(f'  Coordinate ranges:')
    valid_kp = keypoints[mask == 1]
    print(f'    X: [{valid_kp[:, 0].min():.3f}, {valid_kp[:, 0].max():.3f}]')
    print(f'    Y: [{valid_kp[:, 1].min():.3f}, {valid_kp[:, 1].max():.3f}]')
    print(f'    Z: [{valid_kp[:, 2].min():.3f}, {valid_kp[:, 2].max():.3f}]')
else:
    print(f'\nKeypoint file not found: {keypoint_path}')

Total training examples: 19935
Columns: ['VIDEO_ID', 'VIDEO_NAME', 'SENTENCE_ID', 'SENTENCE_NAME', 'START_REALIGNED', 'END_REALIGNED', 'SENTENCE', 'row_duration_sec', 'duration_sec', 'word_count']

Sample: 1Rvc-TbUBJ0_17-5-rgb_front
Text: "And just as safe as you do again with the other brake pad, and it's secure."
Duration: 6.27s

Keypoint file: 1Rvc-TbUBJ0_17-5-rgb_front.npz
  Shape: (T=280 frames, N=116 landmarks, C=3 coords)
  Valid landmarks: 32291 / 32480 (99.4%)
  Coordinate ranges:
    X: [-0.993, 0.924]
    Y: [-1.526, 3.085]
    Z: [-1.061, 0.050]


In [2]:
## Step 2: Build Dataset Manifest

from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Paths
TRAIN_CSV = Path('..') / 'data' / '(final)_how2sign_train_filtered.csv'
KEYPOINTS_DIR = Path('..') / 'data' / 'keypoints_preprocessed' / 'train'

# Load CSV
df = pd.read_csv(TRAIN_CSV, sep='\t')

# Build manifest: list of (sentence_name, text, duration, keypoint_path)
manifest = []
missing = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Building manifest'):
    sentence_name = row['SENTENCE_NAME']
    text = row['SENTENCE']
    duration = row['END_REALIGNED'] - row['START_REALIGNED']
    keypoint_path = KEYPOINTS_DIR / f'{sentence_name}.npz'
    
    if keypoint_path.exists():
        manifest.append({
            'sentence_name': sentence_name,
            'text': text,
            'duration': duration,
            'keypoint_path': str(keypoint_path)
        })
    else:
        missing.append(sentence_name)

print(f'\nDataset manifest:')
print(f'  Total CSV rows: {len(df)}')
print(f'  Found keypoints: {len(manifest)}')
print(f'  Missing keypoints: {len(missing)}')
print(f'  Success rate: {len(manifest)/len(df)*100:.1f}%')

if len(missing) > 0:
    print(f'\nSample missing files (first 5):')
    for name in missing[:5]:
        print(f'  {name}')

Building manifest: 100%|██████████| 19935/19935 [00:04<00:00, 4889.38it/s]


Dataset manifest:
  Total CSV rows: 19935
  Found keypoints: 19935
  Missing keypoints: 0
  Success rate: 100.0%


In [3]:
## Step 4: Setup Qwen3-0.6B-Base Tokenizer

from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base")

print("Tokenizer loaded:")
print(f"  Vocab size: {len(tokenizer)}")
print(f"\nSpecial tokens:")
print(f"  BOS token: {tokenizer.bos_token} (id={tokenizer.bos_token_id})")
print(f"  EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"  PAD token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"  UNK token: {tokenizer.unk_token} (id={tokenizer.unk_token_id})")

# Test tokenization on a sample sentence from dataset
sample_text = manifest[0]['text']
tokens = tokenizer(sample_text, return_tensors='pt')

print(f"\nSample text: \"{sample_text}\"")
print(f"Tokenized:")
print(f"  Input IDs shape: {tokens['input_ids'].shape}")
print(f"  Input IDs: {tokens['input_ids'][0].tolist()[:20]}{'...' if len(tokens['input_ids'][0]) > 20 else ''}")
print(f"  Decoded back: \"{tokenizer.decode(tokens['input_ids'][0])}\"")

# Show token breakdown
print(f"\nToken breakdown (first 10):")
for i, token_id in enumerate(tokens['input_ids'][0][:10].tolist()):
    token_str = tokenizer.decode([token_id])
    print(f"  {i}: {token_id:5d} -> '{token_str}'")

Tokenizer loaded:
  Vocab size: 151669

Special tokens:
  BOS token: None (id=None)
  EOS token: <|endoftext|> (id=151643)
  PAD token: <|endoftext|> (id=151643)
  UNK token: None (id=None)

Sample text: "This is all the you know, take off on the idea of the acanthus leaf."
Tokenized:
  Input IDs shape: torch.Size([1, 19])
  Input IDs: [1986, 374, 678, 279, 498, 1414, 11, 1896, 1007, 389, 279, 4522, 315, 279, 1613, 31229, 355, 15933, 13]
  Decoded back: "This is all the you know, take off on the idea of the acanthus leaf."

Token breakdown (first 10):
  0:  1986 -> 'This'
  1:   374 -> ' is'
  2:   678 -> ' all'
  3:   279 -> ' the'
  4:   498 -> ' you'
  5:  1414 -> ' know'
  6:    11 -> ','
  7:  1896 -> ' take'
  8:  1007 -> ' off'
  9:   389 -> ' on'


In [4]:
## Step 5: Add Custom BOS and PAD Tokens

# Add custom special tokens
num_added = tokenizer.add_special_tokens({
    'bos_token': '<|startoftext|>',
    'pad_token': '<|pad|>'
})

print(f"Added {num_added} new token(s)")
print(f"\nUpdated special tokens:")
print(f"  BOS token: {tokenizer.bos_token} (id={tokenizer.bos_token_id})")
print(f"  EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"  PAD token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"\nVocab size: {len(tokenizer)} (original 151669 + {num_added} new tokens)")

# Test tokenization
sample_text = manifest[0]['text']
tokens = tokenizer(sample_text, add_special_tokens=True, return_tensors='pt')

print(f"\nSample: \"{sample_text}\"")
print(f"Token IDs (first 10): {tokens['input_ids'][0][:10].tolist()}")
print(f"\nNote: BOS/EOS will be added explicitly in Dataset class for decoder")
print(f"      PAD will be used for batching variable-length sequences")

Added 2 new token(s)

Updated special tokens:
  BOS token: <|startoftext|> (id=151669)
  EOS token: <|endoftext|> (id=151643)
  PAD token: <|pad|> (id=151670)

Vocab size: 151671 (original 151669 + 2 new tokens)

Sample: "This is all the you know, take off on the idea of the acanthus leaf."
Token IDs (first 10): [1986, 374, 678, 279, 498, 1414, 11, 1896, 1007, 389]

Note: BOS/EOS will be added explicitly in Dataset class for decoder
      PAD will be used for batching variable-length sequences


In [5]:
## Step 6: Update Dataset to Tokenize Text (with BOS/EOS, max_length, error handling)

import torch
from torch.utils.data import Dataset
import numpy as np

# Max sequence lengths (safety caps to avoid OOM)
MAX_KEYPOINT_FRAMES = 200  # At 20 FPS, 10 seconds (our max is 8s, so plenty of headroom)
MAX_TEXT_TOKENS = 100  # Most sentences well under this (excluding BOS/EOS)

class SignLanguageDataset(Dataset):
    def __init__(self, manifest, tokenizer, max_frames=MAX_KEYPOINT_FRAMES, max_tokens=MAX_TEXT_TOKENS):
        """
        Args:
            manifest: List of dicts with keys: sentence_name, text, duration, keypoint_path
            tokenizer: Pretrained tokenizer (Qwen3)
            max_frames: Max keypoint frames (truncate if longer)
            max_tokens: Max text tokens excluding BOS/EOS (truncate if longer)
        """
        self.manifest = manifest
        self.tokenizer = tokenizer
        self.max_frames = max_frames
        self.max_tokens = max_tokens
    
    def __len__(self):
        return len(self.manifest)
    
    def __getitem__(self, idx):
        sample = self.manifest[idx]
        
        # Load keypoints with error handling
        try:
            data = np.load(sample['keypoint_path'])
            keypoints = data['keypoints'][:, :, :2]  # (T, N, 2) - drop Z
            mask = data['mask']  # (T, N)
        except Exception as e:
            raise RuntimeError(
                f"Failed to load keypoints for {sample['sentence_name']}: {e}\n"
                f"Path: {sample['keypoint_path']}"
            )
        
        # Truncate keypoints if too long
        if keypoints.shape[0] > self.max_frames:
            keypoints = keypoints[:self.max_frames]
            mask = mask[:self.max_frames]
        
        # Convert keypoints to torch tensors
        keypoints = torch.from_numpy(keypoints).float()  # (T, N, 2)
        mask = torch.from_numpy(mask).bool()  # (T, N) - bool for attention masking
        
        # Tokenize text
        text = sample['text']
        token_ids = self.tokenizer.encode(text, add_special_tokens=False)
        
        # Truncate tokens if too long (reserve 2 for BOS/EOS)
        if len(token_ids) > self.max_tokens:
            token_ids = token_ids[:self.max_tokens]
        
        # Add BOS at start, EOS at end
        token_ids = [self.tokenizer.bos_token_id] + token_ids + [self.tokenizer.eos_token_id]
        token_ids = torch.tensor(token_ids, dtype=torch.long)
        
        return {
            'keypoints': keypoints,  # (T, N, 2)
            'keypoint_mask': mask,  # (T, N) bool
            'token_ids': token_ids,  # (L,) where L = text_len + 2 (BOS + EOS)
            'sentence_name': sample['sentence_name']  # For debugging
        }

# Create updated dataset with tokenizer
train_dataset = SignLanguageDataset(manifest, tokenizer)
print(f'Dataset updated: {len(train_dataset)} samples')
print(f'Max keypoint frames: {MAX_KEYPOINT_FRAMES}')
print(f'Max text tokens: {MAX_TEXT_TOKENS} (excluding BOS/EOS)')

# Test: load one sample
sample = train_dataset[0]
print(f"\nSample 0:")
print(f"  Keypoints shape: {sample['keypoints'].shape}")
print(f"  Keypoint mask shape: {sample['keypoint_mask'].shape}, dtype: {sample['keypoint_mask'].dtype}")
print(f"  Token IDs shape: {sample['token_ids'].shape}")
print(f"  Token IDs: {sample['token_ids'][:12].tolist()}...")
print(f"  First token (BOS): {sample['token_ids'][0].item()} (should be {tokenizer.bos_token_id})")
print(f"  Last token (EOS): {sample['token_ids'][-1].item()} (should be {tokenizer.eos_token_id})")
print(f"  Decoded text: \"{tokenizer.decode(sample['token_ids'])}\"")
print(f"\nDuring training:")
print(f"  Decoder input = token_ids[:-1] (BOS to second-to-last)")
print(f"  Labels = token_ids[1:] (second to EOS)")


# ● The 3 dimensions explained:

#   torch.Size([144, 116, 3]) means:
#   - 144 = Number of frames (T) - temporal dimension
#   - 116 = Number of landmarks (N) - 60 face + 14 pose + 42 hands
#   - 3 = Coordinates (x, y, z) for each landmark

#   So shape is (T, N, C) = (frames, landmarks, coordinates)

Dataset updated: 19935 samples
Max keypoint frames: 200
Max text tokens: 100 (excluding BOS/EOS)

Sample 0:
  Keypoints shape: torch.Size([144, 116, 2])
  Keypoint mask shape: torch.Size([144, 116]), dtype: torch.bool
  Token IDs shape: torch.Size([21])
  Token IDs: [151669, 1986, 374, 678, 279, 498, 1414, 11, 1896, 1007, 389, 279]...
  First token (BOS): 151669 (should be 151669)
  Last token (EOS): 151643 (should be 151643)
  Decoded text: "<|startoftext|>This is all the you know, take off on the idea of the acanthus leaf.<|endoftext|>"

During training:
  Decoder input = token_ids[:-1] (BOS to second-to-last)
  Labels = token_ids[1:] (second to EOS)


In [6]:
## Step 7: Create Collate Function for Batching

import torch

def collate_fn(batch, pad_token_id):
    """
    Collate function to batch variable-length samples.
    
    Args:
        batch: List of dicts from Dataset.__getitem__
        pad_token_id: Token ID to use for padding text
        
    Returns:
        Dict with batched tensors:
            keypoints: (B, max_T, N, 2) - padded keypoints
            keypoint_mask: (B, max_T, N) - bool, True=valid, False=padded
            token_ids: (B, max_L) - padded token IDs
            text_attention_mask: (B, max_L) - bool, True=real token, False=padding
    """
    # Extract individual components
    keypoints_list = [item['keypoints'] for item in batch]  # List of (T_i, N, 2)
    keypoint_mask_list = [item['keypoint_mask'] for item in batch]  # List of (T_i, N)
    token_ids_list = [item['token_ids'] for item in batch]  # List of (L_i,)
    
    # Find max lengths in this batch
    max_keypoint_frames = max(kp.shape[0] for kp in keypoints_list)
    max_token_length = max(tokens.shape[0] for tokens in token_ids_list)
    
    batch_size = len(batch)
    num_landmarks = keypoints_list[0].shape[1]  # N = 116
    
    # Initialize padded tensors
    padded_keypoints = torch.zeros(batch_size, max_keypoint_frames, num_landmarks, 2)
    padded_keypoint_mask = torch.zeros(batch_size, max_keypoint_frames, num_landmarks, dtype=torch.bool)
    padded_token_ids = torch.full((batch_size, max_token_length), pad_token_id, dtype=torch.long)
    text_attention_mask = torch.zeros(batch_size, max_token_length, dtype=torch.bool)
    
    # Fill in actual data
    for i in range(batch_size):
        # Keypoints
        T = keypoints_list[i].shape[0]
        padded_keypoints[i, :T] = keypoints_list[i]
        padded_keypoint_mask[i, :T] = keypoint_mask_list[i]
        
        # Token IDs
        L = token_ids_list[i].shape[0]
        padded_token_ids[i, :L] = token_ids_list[i]
        text_attention_mask[i, :L] = True  # Real tokens
    
    return {
        'keypoints': padded_keypoints,  # (B, max_T, N, 2)
        'keypoint_mask': padded_keypoint_mask,  # (B, max_T, N) bool
        'token_ids': padded_token_ids,  # (B, max_L) long
        'text_attention_mask': text_attention_mask,  # (B, max_L) bool
    }

# Create partial function with pad_token_id for DataLoader
from functools import partial
collate_fn_with_tokenizer = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

# Test collate function with a small batch
from torch.utils.data import DataLoader

test_loader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn_with_tokenizer)
batch = next(iter(test_loader))

print("Batch shapes:")
print(f"  Keypoints: {batch['keypoints'].shape}")
print(f"  Keypoint mask: {batch['keypoint_mask'].shape}, dtype: {batch['keypoint_mask'].dtype}")
print(f"  Token IDs: {batch['token_ids'].shape}")
print(f"  Text attention mask: {batch['text_attention_mask'].shape}, dtype: {batch['text_attention_mask'].dtype}")

print(f"\nBatch details:")
print(f"  Batch size: {batch['keypoints'].shape[0]}")
print(f"  Max keypoint frames in batch: {batch['keypoints'].shape[1]}")
print(f"  Max text length in batch: {batch['token_ids'].shape[1]}")
print(f"  Num landmarks: {batch['keypoints'].shape[2]}")

print(f"\nSample token IDs from batch[0]:")
print(f"  First 10: {batch['token_ids'][0, :10].tolist()}")
print(f"  Attention mask first 10: {batch['text_attention_mask'][0, :10].tolist()}")

# What's in the batch (what we just saw):
# keypoints: (4, 160, 116, 2) - encoder input (4 samples, up to 160 frames, 116 landmarks, x/y)
# keypoint_mask: (4, 160, 116) bool - which landmarks are valid
# token_ids: (4, 21) - FULL sequence: [BOS, token1, ..., tokenN, EOS]
# text_attention_mask: (4, 21) bool - which tokens are real vs padding

# 1. ENCODER gets:
# Input: keypoints = (4, 160, 116, 2) *****(It's the MLP output accutally, but we will add that later)
# Mask: keypoint_mask = (4, 160, 116)
# ↓
# Encoder processes this
# ↓
# Output: encoder_hidden_states = (4, 160, d_model)
# # where d_model is encoder's hidden dimension (e.g., 512)

# 2. DECODER gets (teacher forcing):
# # We SPLIT token_ids into two:

# Decoder INPUT (shifted right):
# decoder_input_ids = token_ids[:, :-1]  # (4, 20)
# # = [BOS, token1, token2, ..., tokenN]
# # Remove last token (EOS)

# Decoder CROSS-ATTENDS to:
# encoder_hidden_states = (4, 160, d_model)

# Decoder OUTPUT:
# logits = (4, 20, vocab_size)  # Predictions for each position

# 3. LOSS computed on:
# Predictions: logits (4, 20, vocab_size)
# Targets: token_ids[:, 1:]  # (4, 20)
#         # = [token1, token2, ..., tokenN, EOS]
#         # Remove first token (BOS)

Batch shapes:
  Keypoints: torch.Size([4, 160, 116, 2])
  Keypoint mask: torch.Size([4, 160, 116]), dtype: torch.bool
  Token IDs: torch.Size([4, 21])
  Text attention mask: torch.Size([4, 21]), dtype: torch.bool

Batch details:
  Batch size: 4
  Max keypoint frames in batch: 160
  Max text length in batch: 21
  Num landmarks: 116

Sample token IDs from batch[0]:
  First 10: [151669, 1986, 374, 678, 279, 498, 1414, 11, 1896, 1007]
  Attention mask first 10: [True, True, True, True, True, True, True, True, True, True]


In [7]:
## Step 8: Load Pretrained Decoder with Hybrid LoRA

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch

# Load pretrained decoder in fp16
model_name = "Qwen/Qwen3-0.6B-Base"
decoder = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)

print(f"Loaded: {model_name}")
print(f"Original vocab size: {decoder.config.vocab_size}")

# Resize embeddings for new tokens (BOS, PAD)
decoder.resize_token_embeddings(len(tokenizer))
print(f"Resized to: {len(tokenizer)} tokens")



Loaded: Qwen/Qwen3-0.6B-Base
Original vocab size: 151936
Resized to: 151671 tokens


In [8]:
print(decoder)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151671, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [9]:
for name, _ in decoder.named_parameters():
    print(name)

model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.v_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.self_attn.q_norm.weight
model.layers.0.self_attn.k_norm.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.v_proj.weight
model.layers.1.self_attn.o_proj.weight
model.layers.1.self_attn.q_norm.weight
model.layers.1.self_attn.k_norm.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.self_attn.q_proj.weight
model.layers.2.self_attn.k_proj.weight
model.layers.2.self_attn.v_proj.weight
model.l

In [10]:
for name, module in decoder.named_modules():
    print(f"{name}: {type(module).__name__}")

: Qwen3ForCausalLM
model: Qwen3Model
model.embed_tokens: Embedding
model.layers: ModuleList
model.layers.0: Qwen3DecoderLayer
model.layers.0.self_attn: Qwen3Attention
model.layers.0.self_attn.q_proj: Linear
model.layers.0.self_attn.k_proj: Linear
model.layers.0.self_attn.v_proj: Linear
model.layers.0.self_attn.o_proj: Linear
model.layers.0.self_attn.q_norm: Qwen3RMSNorm
model.layers.0.self_attn.k_norm: Qwen3RMSNorm
model.layers.0.mlp: Qwen3MLP
model.layers.0.mlp.gate_proj: Linear
model.layers.0.mlp.up_proj: Linear
model.layers.0.mlp.down_proj: Linear
model.layers.0.mlp.act_fn: SiLUActivation
model.layers.0.input_layernorm: Qwen3RMSNorm
model.layers.0.post_attention_layernorm: Qwen3RMSNorm
model.layers.1: Qwen3DecoderLayer
model.layers.1.self_attn: Qwen3Attention
model.layers.1.self_attn.q_proj: Linear
model.layers.1.self_attn.k_proj: Linear
model.layers.1.self_attn.v_proj: Linear
model.layers.1.self_attn.o_proj: Linear
model.layers.1.self_attn.q_norm: Qwen3RMSNorm
model.layers.1.self_a

In [11]:
## Step 8 (continued): Apply Hybrid LoRA Configuration

# Configure LoRA with comprehensive coverage
lora_config = LoraConfig(
    r=32,  # Low-rank dimension (higher = more capacity, more memory)
    lora_alpha=64,  # Scaling factor (typically 2*r)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Self-attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
    modules_to_save=["embed_tokens", "lm_head"],  # Fully trainable
    lora_dropout=0.1,
    bias="none",  # Qwen3 has no bias parameters
    task_type="CAUSAL_LM",
    use_rslora=True 
)

# Apply LoRA
decoder = get_peft_model(decoder, lora_config)

# Manually enable norm layers (not covered by LoRA or modules_to_save)
norm_params_count = 0
for name, param in decoder.named_parameters():
    if 'norm' in name:
        param.requires_grad = True
        norm_params_count += param.numel()

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules (LoRA): {lora_config.target_modules}")
print(f"  Modules to save (fully trainable): {lora_config.modules_to_save}")
print(f"  Norm layers: manually set to trainable ({norm_params_count / 1e6:.2f}M params)")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Bias: {lora_config.bias}")

# Print trainable parameters
print("\nTrainable parameters breakdown:")
decoder.print_trainable_parameters()

print(f"\nDecoder architecture:")
print(f"  Hidden size (d_model): {decoder.config.hidden_size}")
print(f"  Num layers: {decoder.config.num_hidden_layers}")
print(f"  Num attention heads: {decoder.config.num_attention_heads}")

print(f"\nHybrid LoRA strategy:")
print(f"  ✓ LoRA adapters: Self-attention + MLP (low-rank, memory efficient)")
print(f"  ✓ Fully trainable: Embeddings, lm_head, all norm layers")
print(f"  ✓ Cross-attention: Will be fully trainable when added (max learning capacity)")
print(f"  → Estimated VRAM savings: ~3+ GB vs full fine-tuning")

C:\Users\mamou\AppData\Roaming\Python\Python312\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


LoRA Configuration:
  Rank (r): 32
  Alpha: 64
  Target modules (LoRA): {'up_proj', 'k_proj', 'v_proj', 'down_proj', 'q_proj', 'gate_proj', 'o_proj'}
  Modules to save (fully trainable): ['embed_tokens', 'lm_head']
  Norm layers: manually set to trainable (0.07M params)
  Dropout: 0.1
  Bias: none

Trainable parameters breakdown:
trainable params: 330,872,832 || all params: 926,585,856 || trainable%: 35.7088

Decoder architecture:
  Hidden size (d_model): 1024
  Num layers: 28
  Num attention heads: 16

Hybrid LoRA strategy:
  ✓ LoRA adapters: Self-attention + MLP (low-rank, memory efficient)
  ✓ Fully trainable: Embeddings, lm_head, all norm layers
  ✓ Cross-attention: Will be fully trainable when added (max learning capacity)
  → Estimated VRAM savings: ~3+ GB vs full fine-tuning


In [12]:
## Step 9: Build MLP Projection Layer

import torch
import torch.nn as nn

class KeypointProjection(nn.Module):
    def __init__(self, num_landmarks=116, coord_dim=2, hidden_dim=512, d_model=512):
        """
        Projects flattened keypoints to encoder dimension.
        
        Args:
            num_landmarks: Number of landmarks (116)
            coord_dim: Coordinates per landmark (2: x, y)
            hidden_dim: Hidden layer size
            d_model: Output dimension (encoder d_model = 512)
        """
        super().__init__()
        
        input_dim = num_landmarks * coord_dim  # 116 * 2 = 232
        
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, d_model),
            nn.LayerNorm(d_model)  # Normalize output for stable encoder input
        )
        
        self.input_dim = input_dim
        self.d_model = d_model
    
    def forward(self, keypoints):
        """
        Args:
            keypoints: (B, T, N, 2) where N=116, 2=x,y
            
        Returns:
            (B, T, d_model) projected representations
        """
        B, T, N, C = keypoints.shape
        
        # Flatten landmarks: (B, T, N, C) -> (B, T, N*C)
        x = keypoints.reshape(B, T, -1)  # (B, T, 232)
        
        # Project to d_model
        x = self.projection(x)  # (B, T, 512)
        
        return x

# Create projection layer
projection = KeypointProjection(num_landmarks=116, coord_dim=2, d_model=512)

# Count parameters
proj_params = sum(p.numel() for p in projection.parameters())
print(f"MLP Projection Layer:")
print(f"  Architecture: Linear(232→512) → ReLU → Dropout → Linear(512→512) → LayerNorm")
print(f"  Input: (B, T, 116, 2) -> flatten to (B, T, 232)")
print(f"  Output: (B, T, 512) - normalized")
print(f"  Parameters: {proj_params / 1e6:.2f}M")

# Test with sample batch
test_keypoints = batch['keypoints']  # (4, 160, 116, 2)
test_output = projection(test_keypoints)

print(f"\nTest:")
print(f"  Input shape: {test_keypoints.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Output dtype: {test_output.dtype}")
print(f"  Output mean: {test_output.mean().item():.4f}, std: {test_output.std().item():.4f}")
print(f"  (LayerNorm ensures stable distribution for encoder input)")

MLP Projection Layer:
  Architecture: Linear(232→512) → ReLU → Dropout → Linear(512→512) → LayerNorm
  Input: (B, T, 116, 2) -> flatten to (B, T, 232)
  Output: (B, T, 512) - normalized
  Parameters: 0.38M

Test:
  Input shape: torch.Size([4, 160, 116, 2])
  Output shape: torch.Size([4, 160, 512])
  Output dtype: torch.float32
  Output mean: -0.0000, std: 0.9982
  (LayerNorm ensures stable distribution for encoder input)
